# When is passive exposure worth accepting?

**Later candidate · B8/C1–C8.** Test whether state selection, fair-value skew and inventory response
improve passive economics. Owner: researcher. Design refreshed 2026-09-23; awaiting evidence.

Follow the taking, shock and cross-venue information candidates. Promotion requires
trade data and credible execution calibration. Begin with the same sample and risk budget as the aggressive studies and retain
the [protocol](../docs/research/study-protocol.md).

**Planned, not started:** research begins only after the infrastructure passes QR onboarding
acceptance. See the [QR delivery milestones](../docs/research/program.md#post-infrastructure-research-milestones).

## 1. State the economic hypothesis

The signal may be more useful for avoiding toxic fills or managing passive exposure than
for paying for an aggressive round trip. Compare a constrained passive baseline with a
toxic-state filter, signal-based quote skew and inventory adjustment **added separately**.

The relevant distinction is `E[return | signal]` versus `E[return | signal, our order fills]`.
Filling is selective: favorable-looking quotes can fill precisely when the next price move
is adverse. A high fill rate and positive unconditional markout do not establish value.

## 2. Assemble execution evidence

Use received books, public trades and the firm's own genuine order-lifecycle records when
available. Retain order placement/acknowledgment, changes, cancellations, fills and timestamps.
Specify what is unobserved: hidden liquidity, queue priority, order age outside feed depth,
trades missed by aggregation, and clock uncertainty.

Public replay can screen a policy under a stated model. It cannot prove attainable passive
fills. A touch is not automatically a fill; a level reduction is not automatically a trade.
Order IDs improve some observations but do not by themselves establish our priority or fills.

Estimate fill-before-adverse-move/timeout jointly with post-fill outcomes. Mark censored
orders and distinguish no-fill from a missing order record. Calibrate with authorized
small-order observations before scaling; this notebook does not send orders.

## 3. Freeze quoting, cancellation and inventory policies

| Component | Research choice |
| --- | --- |
| Placement | Quote distance, side, quantity, lifetime and maximum outstanding orders |
| Selection | Quote only in admitted states; retain rejected opportunities |
| Skew | Translate a simple fair-value estimate into price/size changes |
| Inventory | Limits, side suppression, quote skew and liquidation rule |
| Cancellation | Delay, in-flight fills, repricing and loss of queue priority |
| Fill mechanics | Partial fills and conservative/plausible queue scenarios |
| Exit | Marked inventory versus realized liquidation; unwind spread/slippage and financing |

Match capital, risk and opportunity sets. Compare take/post/do-nothing on common decision
states when execution evidence permits it. Test fixed horizon, signal decay/reversal and
inventory-triggered exits only as declared variants. Retain tail losses and opportunity cost
of missed fills; do not evaluate just spread captured on successful fills.

## 4. Supply daily outcomes under plausible execution scenarios

The table summarizes a retained evaluation; it is not a queue simulator. Use one instrument,
quote currency and fixed capital/risk specification. `scenario` identifies a **joint**
fill, cancellation, delay and adverse-selection model, not an independently varied fill rate.
Every scenario has the same days and policies. Include inactive/zero-fill days.

`filled_orders` counts orders with any fill; `partial_fill_orders` is a subset.
`filled_quantity_base` sums executed base quantity. `fill_markout_value_quote` sums
signed post-fill midprice change × filled quantity at one predeclared horizon (positive
is favorable to the filled side); it is a diagnostic, not net P&L. Dividing by filled
quantity produces quote units per base unit, not basis points.

`net_pnl_quote` already includes fees, financing and `unwind_cost_quote`; do not subtract
unwind cost again. Retain complete inventory paths and the economics of unfilled quantities.

In [ ]:
# standard library imports
from math import isfinite

# third party library imports
import pandas as pd

study_ref = ""
policies = ["passive_baseline", "toxicity_filter", "fair_value_skew", "inventory_response"]
daily_results: list[dict[str, object]] = []
columns = [
    "day",
    "scenario",
    "policy",
    "opportunity_set_id",
    "quoted_orders",
    "filled_orders",
    "partial_fill_orders",
    "filled_quantity_base",
    "fill_markout_value_quote",
    "net_pnl_quote",
    "unwind_cost_quote",
    "max_abs_inventory_quote",
]
daily = pd.DataFrame(daily_results, columns=columns)

In [ ]:
if daily.empty:
    print("AWAITING EVIDENCE — no passive fill or profitability claim is established.")
else:
    if (
        not study_ref.strip()
        or daily.isna().any().any()
        or daily.duplicated(["day", "scenario", "policy"]).any()
    ):
        raise ValueError("Supply a retained study and complete unique day/scenario/policy rows.")
    expected_days = set(daily.day)
    for _, scenario_rows in daily.groupby("scenario"):
        if set(scenario_rows.day) != expected_days:
            raise ValueError("Scenarios must cover the same days.")
    if (daily.groupby("day").opportunity_set_id.nunique() != 1).any():
        raise ValueError("Execution scenarios must use the same daily opportunity set.")
    for _, matched in daily.groupby(["day", "scenario"]):
        if set(matched.policy) != set(policies) or matched.opportunity_set_id.nunique() != 1:
            raise ValueError("Compare all policies on a common opportunity set.")
    for field in columns[4:]:
        daily[field] = pd.to_numeric(daily[field], errors="raise")
    if (
        (
            daily[
                [
                    "quoted_orders",
                    "filled_orders",
                    "partial_fill_orders",
                    "filled_quantity_base",
                    "max_abs_inventory_quote",
                ]
            ]
            < 0
        )
        .any()
        .any()
    ):
        raise ValueError("Counts, filled quantity and absolute inventory cannot be negative.")
    if (daily.filled_orders > daily.quoted_orders).any() or (
        daily.partial_fill_orders > daily.filled_orders
    ).any():
        raise ValueError("Fill counts must respect the quoted/filled/partial subsets.")
    if not daily[columns[4:]].map(isfinite).all().all():
        raise ValueError("Execution outcomes must be finite.")
    if (daily[["quoted_orders", "filled_orders", "partial_fill_orders"]] % 1 != 0).any().any():
        raise ValueError("Order counts must be integers.")
    if ((daily.filled_orders == 0) != (daily.filled_quantity_base == 0)).any():
        raise ValueError("Filled orders and positive filled quantity must agree.")
    if ((daily.filled_quantity_base == 0) & (daily.fill_markout_value_quote != 0)).any():
        raise ValueError("A zero-fill day has zero filled markout value.")
    summary = daily.groupby(["scenario", "policy"])[columns[4:11]].sum()
    summary["fill_rate_per_order"] = summary.filled_orders / summary.quoted_orders.where(
        summary.quoted_orders > 0
    )
    summary["markout_quote_per_base"] = (
        summary.fill_markout_value_quote
        / summary.filled_quantity_base.where(summary.filled_quantity_base > 0)
    )
    peak = daily.groupby(["scenario", "policy"]).max_abs_inventory_quote.max()
    summary["peak_inventory_quote"] = peak
    paired = daily.pivot(index=["day", "scenario"], columns="policy", values="net_pnl_quote")
    improvement = paired.subtract(paired["passive_baseline"], axis=0)
    display(summary)
    display(improvement.rename_axis(columns="daily net improvement over passive_baseline"))

## 5. Challenge fills, risk and attribution

Compare partial fills, fill-conditioned markouts, cancellation races, inventory duration,
forced unwind cost and daily net P&L. Check whether a filter improves economics or simply
changes directional exposure. Keep comparisons matched on risk and capital. A low fill
rate is not inherently bad; it matters together with conditional value and opportunity cost.

Evaluate conservative and other plausible scenarios jointly. Do not optimize a fill model
to match a desired P&L. Inspect tail losses and dependence across orders/days, calibration
errors and discrepancies between replay and genuine order observations.

If plausible scenarios change the sign of profit, keep the result unvalidated. If net gains
disappear with inventory/unwind cost, reject or narrow. Empirical execution calibration
and operational qualification are additional requirements before scaling.

## 6. Decision and next experiment

Complete the [memo](../docs/research/study-protocol.md#decision-memo): matched baseline deltas, fill-model
sensitivity, calibration sample, markouts, inventory paths, tail losses and unresolved
observability. **Current conclusion: insufficient evidence.**

After a credible local baseline, D2 asks whether a second venue's already-received state
improves quote selection/skew/cancellation. Keep currency and contract differences explicit.
D4 adds separate questions of prefunding, asymmetric fills, hedge delay and rebalancing;
it is not a small extension of this passive study.